In [5]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [6]:
MODEL_FILE = "model.pkl"
PIPELINE_FILE = "pipeline.pkl"


def build_pipeline(num_attribs, cat_attribs):

    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", cat_pipeline, cat_attribs)
    ])

    return full_pipeline

In [11]:
if not os.path.exists(MODEL_FILE):

    print("Training model...")

    # Load dataset
    housing = pd.read_csv("housing.csv")

    # Create income categories for stratified sampling
    housing["income_cat"] = pd.cut(
        housing["median_income"],
        bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
        labels=[1, 2, 3, 4, 5]
    )

    # Stratified train/test split
    split = StratifiedShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    for train_index, test_index in split.split(
        housing,
        housing["income_cat"]
    ):
        train_set = housing.loc[train_index].copy()
        test_set = housing.loc[test_index].copy()

    # Remove income_cat
    train_set.drop("income_cat", axis=1, inplace=True)
    test_set.drop("income_cat", axis=1, inplace=True)

    # Separate labels and features
    housing_labels = train_set["median_house_value"].copy()
    housing_features = train_set.drop(
        "median_house_value",
        axis=1
    )

    test_labels = test_set["median_house_value"].copy()
    test_features = test_set.drop(
        "median_house_value",
        axis=1
    )

    # Numerical and categorical attributes
    num_attribs = housing_features.drop(
        "ocean_proximity",
        axis=1
    ).columns.tolist()

    cat_attribs = ["ocean_proximity"]

    # Build preprocessing pipeline
    pipeline = build_pipeline(
        num_attribs,
        cat_attribs
    )

    # Transform training data
    housing_prepared = pipeline.fit_transform(
        housing_features
    )

    # Transform test data
    test_prepared = pipeline.transform(
        test_features
    )

    # Create Random Forest model
    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    # Train model
    model.fit(
        housing_prepared,
        housing_labels
    )

    predictions = model.predict(test_prepared)

    mae = mean_absolute_error(
        test_labels,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            test_labels,
            predictions
        )
    )

    r2 = r2_score(
        test_labels,
        predictions
    )

    print("\nModel Evaluation")
    print("----------------")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", r2)

    joblib.dump(
        model,
        MODEL_FILE
    )

    joblib.dump(
        pipeline,
        PIPELINE_FILE
    )

    print("\nModel trained and saved.")
    print("Created:", MODEL_FILE)
    print("Created:", PIPELINE_FILE)

else:

    print("Loading saved model...")

    # Load model and pipeline
    model = joblib.load(MODEL_FILE)
    pipeline = joblib.load(PIPELINE_FILE)

    # Load new house data
    input_data = pd.read_csv("input.csv")

    # Transform input data
    transformed_input = pipeline.transform(
        input_data
    )

    # Make predictions
    predictions = model.predict(
        transformed_input
    )

    # Add predictions to input data
    input_data["median_house_value"] = predictions

    # Display predictions
    print("\nHouse Price Predictions:")
    print(input_data)

    # Save predictions
    input_data.to_csv(
        "predictions.csv",
        index=False
    )

    print("\nPredictions saved to predictions.csv")


Training model...

Model Evaluation
----------------
MAE : 30929.476097383722
RMSE: 47197.66824186381
R²  : 0.8290804707970139

Model trained and saved.
Created: model.pkl
Created: pipeline.pkl
